# Community voting for uncertain snake sightings

A reviewer can vote for a species or stop at genus. Each reviewer has one active vote per sighting. The ballot returns species consensus, genus consensus, or pending.

[Van Horn et al. (2018), *Lean Multiclass Crowdsourcing*](https://openaccess.thecvf.com/content_cvpr_2018/CameraReady/1324.pdf), §3.1.2–3.1.3, motivates factoring a vote along the taxonomy and using only the levels the reviewer reported. [Wang, Lauha and Dunson (2024), *Bayesian modeling of multi-species labeling errors in ecological studies*](https://arxiv.org/html/2406.15844v1#S2.SS4), motivates allowing annotation accuracy to vary across reviewers. The model below adapts these ideas to one snake per sighting and uses uncalibrated role-level accuracies in the absence of reviewer history.

The latent label is a species in the project catalog, an unlisted species within a known genus, or an unlisted genus. The prior is uniform over genus options, then uniform over species options within each known genus. Model predictions are excluded from the prior for these low-confidence sightings. At each reported rank, a reviewer has the role-specific probability of giving the correct label and spreads the remaining probability uniformly across the other options. A species vote under the wrong genus is uniform over that reported genus's species options. The likelihood multiplies conditionally independent votes.

Sum species posterior mass within each genus to get the genus posterior. Select the most specific rank with a unique winner, posterior of at least 0.85, and at least 3 points of direct or rolled-up support for that winner. Unlisted alternatives can absorb posterior mass and keep a result pending.

In [ ]:
import csv
from collections import Counter
from math import exp, fsum, isclose, log
from pathlib import Path

ROLE_WEIGHT = {"citizen": 1, "verified_rescuer": 2, "expert": 3}
# Each pair gives genus accuracy and species accuracy conditional on the correct genus.
ROLE_ACCURACY = {
    "citizen": (0.65, 0.60),
    "verified_rescuer": (0.80, 0.75),
    "expert": (0.90, 0.85),
}
MIN_POSTERIOR = 0.85
MIN_SUPPORT = 3

catalog_path = Path("data/classes.csv")
if not catalog_path.is_file():
    catalog_path = Path("../data/classes.csv")
with catalog_path.open(encoding="utf-8", newline="") as catalog_file:
    SPECIES_TO_GENUS = {
        row["species"]: row["genus"] for row in csv.DictReader(catalog_file)
    }

In [ ]:
def label_likelihood(reported, truth, accuracy, option_count):
    """Categorical response model with uniform errors."""
    if reported == truth:
        return accuracy
    return (1 - accuracy) / (option_count - 1)


def consensus_label(probabilities, support):
    """Return a named label only if it clears both evidence gates."""
    candidate = max(probabilities, key=probabilities.get)
    if candidate is None or support[candidate] < MIN_SUPPORT:
        return None

    confidence = probabilities[candidate]
    tied = any(
        label != candidate and isclose(probability, confidence, rel_tol=1e-12)
        for label, probability in probabilities.items()
    )
    if confidence >= MIN_POSTERIOR and not tied:
        return candidate
    return None

In [ ]:
class SightingBallot:
    """Collect and aggregate active votes for one sighting."""

    def __init__(self, reviewer_roles, species_to_genus=SPECIES_TO_GENUS):
        self._reviewer_roles = dict(reviewer_roles)
        self._species_to_genus = dict(species_to_genus)
        if not self._species_to_genus:
            raise ValueError("The species catalog is empty.")
        if any(role not in ROLE_WEIGHT for role in self._reviewer_roles.values()):
            raise ValueError("Every reviewer must have a recognized role.")

        # None represents an unlisted alternative at each rank.
        self._species_options = {None: [None]}
        for species, genus in self._species_to_genus.items():
            self._species_options.setdefault(genus, [None]).append(species)
        self._votes = {}

    def cast_vote(self, reviewer_id, rank, label):
        """Validate and save the reviewer's current choice."""
        if reviewer_id not in self._reviewer_roles:
            raise ValueError(f"Unknown reviewer: {reviewer_id!r}")
        if rank == "species" and label in self._species_to_genus:
            genus, species = self._species_to_genus[label], label
        elif rank == "genus" and label is not None and label in self._species_options:
            genus, species = label, None
        else:
            raise ValueError(f"Invalid {rank!r} vote for {label!r}")

        self._votes[reviewer_id] = (genus, species)

    def result(self):
        weighted_genus, weighted_species = Counter(), Counter()
        vote_evidence = []
        for reviewer_id, (genus, species) in self._votes.items():
            role = self._reviewer_roles[reviewer_id]
            weight = ROLE_WEIGHT[role]
            weighted_genus[genus] += weight
            if species is not None:
                weighted_species[species] += weight
            vote_evidence.append((genus, species, *ROLE_ACCURACY[role]))

        genus_count = len(self._species_options)
        log_scores = {}
        for true_genus, species_options in self._species_options.items():
            log_prior = -log(genus_count) - log(len(species_options))
            for true_species in species_options:
                log_score = log_prior
                for genus, species, genus_accuracy, species_accuracy in vote_evidence:
                    log_score += log(label_likelihood(
                        genus, true_genus, genus_accuracy, genus_count
                    ))
                    if species is not None:
                        option_count = len(self._species_options[genus])
                        likelihood = 1 / option_count
                        if genus == true_genus:
                            likelihood = label_likelihood(
                                species, true_species, species_accuracy, option_count
                            )
                        log_score += log(likelihood)
                log_scores[true_genus, true_species] = log_score

        # Shift log scores before exponentiating to preserve numerical stability.
        offset = max(log_scores.values())
        masses = {taxon: exp(score - offset) for taxon, score in log_scores.items()}
        total_mass = fsum(masses.values())
        species_posterior = {taxon: mass / total_mass for taxon, mass in masses.items()}
        genus_posterior = {
            genus: fsum(species_posterior[genus, species] for species in species_options)
            for genus, species_options in self._species_options.items()
        }
        named_species_posterior = {
            species: species_posterior[genus, species]
            for species, genus in self._species_to_genus.items()
        }
        # Leave unlisted species as individual alternatives when checking the winner.
        weighted_paths = Counter({
            (self._species_to_genus[species], species): weight
            for species, weight in weighted_species.items()
        })
        species_path = consensus_label(species_posterior, weighted_paths)
        genus_label = consensus_label(genus_posterior, weighted_genus)

        status, rank, label, confidence = "pending", None, None, None
        if species_path is not None:
            status, rank, label = "species_consensus", "species", species_path[1]
            confidence = species_posterior[species_path]
        elif genus_label is not None:
            status, rank, label = "genus_consensus", "genus", genus_label
            confidence = genus_posterior[genus_label]

        return {
            "status": status,
            "rank": rank,
            "label": label,
            "posterior_confidence": confidence,
            "reviewer_count": len(self._votes),
            "support_weight": {"genus": dict(weighted_genus), "species": dict(weighted_species)},
            "genus_posterior": genus_posterior,
            "species_posterior": named_species_posterior,
            "unlisted_species_posterior": {
                genus: species_posterior[genus, None]
                for genus in self._species_options if genus is not None
            },
        }

## Voting scenarios

These checks use catalog taxa and synthetic ballots. The posterior measures evidence under the stated assumptions. Its empirical calibration requires labeled sightings with reviewer votes.

In [ ]:
REVIEWERS = {
    "citizen_1": "citizen",
    "citizen_2": "citizen",
    "citizen_3": "citizen",
    "rescuer_1": "verified_rescuer",
    "rescuer_2": "verified_rescuer",
    "expert_1": "expert",
}

scenarios = [
    ("No votes", [], "pending", None),
    ("Two citizens below minimum support", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja naja"),
    ], "pending", None),
    ("Three agreeing citizens", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja naja"),
        ("citizen_3", "species", "Naja naja"),
    ], "species_consensus", "Naja naja"),
    ("Same genus, different species", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja kaouthia"),
        ("citizen_3", "genus", "Naja"),
    ], "genus_consensus", "Naja"),
    ("Direct genus vote from one expert", [
        ("expert_1", "genus", "Naja"),
    ], "genus_consensus", "Naja"),
    ("One expert species vote resolves only the genus", [
        ("expert_1", "species", "Naja naja"),
    ], "genus_consensus", "Naja"),
    ("Equal support for different genera", [
        ("rescuer_1", "genus", "Naja"),
        ("rescuer_2", "genus", "Ptyas"),
    ], "pending", None),
    ("Expert and rescuer outweigh one conflicting citizen", [
        ("expert_1", "species", "Naja naja"),
        ("rescuer_1", "species", "Naja naja"),
        ("citizen_1", "species", "Ptyas mucosa"),
    ], "species_consensus", "Naja naja"),
]

for name, votes, expected_status, expected_label in scenarios:
    ballot = SightingBallot(REVIEWERS)
    for vote in votes:
        ballot.cast_vote(*vote)
    outcome = ballot.result()
    assert (outcome["status"], outcome["label"]) == (expected_status, expected_label), (
        name, outcome["status"], outcome["label"]
    )
    assert outcome["reviewer_count"] == len(votes)
    assert isclose(fsum(outcome["genus_posterior"].values()), 1)
    total_species_mass = (
        fsum(outcome["species_posterior"].values())
        + fsum(outcome["unlisted_species_posterior"].values())
        + outcome["genus_posterior"][None]
    )
    assert isclose(total_species_mass, 1)
    for genus, genus_mass in outcome["genus_posterior"].items():
        if genus is not None:
            species_mass = fsum(
                probability for species, probability in outcome["species_posterior"].items()
                if SPECIES_TO_GENUS[species] == genus
            )
            assert isclose(species_mass + outcome["unlisted_species_posterior"][genus], genus_mass)

    confidence = outcome["posterior_confidence"]
    detail = "" if confidence is None else f" -> {outcome['label']} ({confidence:.3f})"
    print(f"{name}: {outcome['status']}{detail}")

In [ ]:
ballot = SightingBallot(REVIEWERS)
ballot.cast_vote("expert_1", "genus", "Naja")
original = ballot.result()
ballot.cast_vote("expert_1", "genus", "Naja")
assert ballot.result() == original

ballot.cast_vote("expert_1", "genus", "Ptyas")
changed = ballot.result()
assert changed["label"] == "Ptyas"
assert changed["reviewer_count"] == 1
assert changed["support_weight"]["genus"] == {"Ptyas": 3}
assert changed["support_weight"]["species"] == {}

invalid_votes = [
    ("unregistered", "genus", "Naja"),
    ("expert_1", "family", "Elapidae"),
    ("expert_1", "species", "Naja"),
    ("expert_1", "genus", None),
]
for vote in invalid_votes:
    try:
        ballot.cast_vote(*vote)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Accepted an invalid vote: {vote!r}")
assert ballot.result() == changed

# A genus-only vote preserves the relative probabilities within that genus.
assert isclose(original["species_posterior"]["Naja naja"],
               original["species_posterior"]["Naja kaouthia"])
assert isclose(original["species_posterior"]["Naja naja"],
               original["unlisted_species_posterior"]["Naja"])

assert consensus_label({"Naja": 0.85, "Ptyas": 0.15}, Counter(Naja=3)) == "Naja"
assert consensus_label({"Naja": 0.849, "Ptyas": 0.151}, Counter(Naja=3)) is None
assert consensus_label({"Naja": 0.95, "Ptyas": 0.05}, Counter(Naja=2)) is None
print("Vote replacement, repeat submissions, invalid votes, genus-only evidence,")
print("and threshold boundaries passed.")